# Neural Networks – 3rd Assignment

An MLP encoder-decoder in PyTorch that takes an MNIST digit x and generates an image of the next digit, x+1. A separate CNN (the "judge") checks how many of the generated digits it can recognise.

The notebook also has the fixed-target experiment and the comparison with PCA.

STANDARD MLP AUTOENCODER (ANY OTHER VERSION CAN BE RUN BY CHANGING THE PARAMETERS WHERE I LEFT A COMMENT)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, SubsetRandomSampler, Dataset, random_split
from torchvision import datasets, transforms
from sklearn.model_selection import KFold
import numpy as np
import random
import time
import matplotlib.pyplot as plt

BATCH_SIZE = 256
NUM_WORKERS = 2
LEARNING_RATE = 1e-3
AE_EPOCHS = 20
REFIT_EPOCHS = 50
JUDGE_EPOCHS = 5
K_FOLDS = 5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ==========================================
# 1. DATASETS
# ==========================================
transform = transforms.ToTensor()
raw_train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
raw_test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

class NextDigitDataset(Dataset):
    def __init__(self, original_dataset):
        self.dataset = original_dataset
        self.buckets = {i: [] for i in range(10)}
        for idx in range(len(original_dataset)):
            _, label = original_dataset[idx]
            self.buckets[label].append(idx)

    def __getitem__(self, index):
        img_input, label_input = self.dataset[index]
        label_target = (label_input + 1) % 10
        random_target_idx = random.choice(self.buckets[label_target])
        img_target, _ = self.dataset[random_target_idx]
        return img_input, img_target, label_input, label_target

    def __len__(self):
        return len(self.dataset)

full_train_dataset = NextDigitDataset(raw_train_dataset)
test_dataset_wrapper = NextDigitDataset(raw_test_dataset)

# ==========================================
# 2. MODEL: STANDARD MLP
# ==========================================
class NextDigitMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # STANDARD MODEL: 784 -> 512 -> 256 -> 128 FOR ANY OTHER EXPERIMENT I JUST CHANGE THE NUMBER OF NEURONS
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 512), nn.ReLU(), #IN THE TANH EXPERIMENT I USE TANH INSTEAD OF RELU
            nn.Linear(512, 256), nn.ReLU(),
            nn.Linear(256, 128)
        )
        self.decoder = nn.Sequential(
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, 512), nn.ReLU(),
            nn.Linear(512, 784), nn.Sigmoid()
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded.view(-1, 1, 28, 28)

def get_model():
    model = NextDigitMLP().to(device)
    optimizer = Adam(model.parameters(), lr=LEARNING_RATE)

    # --- MSE LOSS ---IN THE OTHER EXPERIMENTS I JUST SWITCH IT TO BCE LOSS
    criterion = nn.MSELoss()

    return model, optimizer, criterion

# ==========================================
# 3. CROSS-VALIDATION
# ==========================================
kfold = KFold(n_splits=K_FOLDS, shuffle=True, random_state=42)

print(f"\n{'='*40}")
print(f"STARTING {K_FOLDS}-FOLD CV (STANDARD MLP + MSE LOSS)")
print(f"{'='*40}")

fold_results = []

for fold, (train_ids, val_ids) in enumerate(kfold.split(raw_train_dataset)):
    print(f"\n>>> FOLD {fold + 1}/{K_FOLDS}")
    start_time = time.time()

    train_subsampler = SubsetRandomSampler(train_ids)
    val_subsampler = SubsetRandomSampler(val_ids)

    train_loader = DataLoader(full_train_dataset, batch_size=BATCH_SIZE, sampler=train_subsampler,
                              num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
    val_loader = DataLoader(full_train_dataset, batch_size=BATCH_SIZE, sampler=val_subsampler,
                            num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

    model, optimizer, criterion = get_model()
    scheduler = CosineAnnealingLR(optimizer, T_max=AE_EPOCHS)# WHEN I CHANGE IT I WRITE scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5)

    for epoch in range(1, AE_EPOCHS + 1):
        model.train()
        train_loss = 0.0
        for img_in, img_target, _, _ in train_loader:
            img_in, img_target = img_in.to(device), img_target.to(device)
            optimizer.zero_grad()
            outputs = model(img_in)
            loss = criterion(outputs, img_target)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * img_in.size(0)

        avg_train_loss = train_loss / len(train_ids)
        scheduler.step()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for img_in, img_target, _, _ in val_loader:
                img_in, img_target = img_in.to(device), img_target.to(device)
                outputs = model(img_in)
                loss = criterion(outputs, img_target)
                val_loss += loss.item() * img_in.size(0)

        avg_val_loss = val_loss / len(val_ids)

        if epoch % 5 == 0:
            print(f"    Epoch {epoch}/{AE_EPOCHS} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    fold_results.append(avg_val_loss)
    print(f"  >> Fold {fold+1} Finished. Time: {time.time()-start_time:.1f}s")

print(f"\nAVERAGE CV LOSS: {np.mean(fold_results):.4f}")

# ==========================================
# 4. REFIT
# ==========================================
print(f"\n{'='*40}")
print("REFIT: Training Final Model")
print(f"{'='*40}")

train_size = int(0.9 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size
refit_train_set, refit_val_set = random_split(full_train_dataset, [train_size, val_size])

refit_train_loader = DataLoader(refit_train_set, batch_size=BATCH_SIZE, shuffle=True,
                                num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
refit_val_loader = DataLoader(refit_val_set, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

final_model, final_optimizer, final_criterion = get_model()
final_scheduler = CosineAnnealingLR(final_optimizer, T_max=REFIT_EPOCHS)

history_train_loss = []
history_val_loss = []

refit_start_time = time.time()

for epoch in range(1, REFIT_EPOCHS + 1):
    final_model.train()
    train_loss = 0.0
    for img_in, img_target, _, _ in refit_train_loader:
        img_in, img_target = img_in.to(device), img_target.to(device)
        final_optimizer.zero_grad()
        outputs = final_model(img_in)
        loss = final_criterion(outputs, img_target)
        loss.backward()
        final_optimizer.step()
        train_loss += loss.item() * img_in.size(0)

    avg_train_loss = train_loss / len(refit_train_set)
    history_train_loss.append(avg_train_loss)
    final_scheduler.step()

    final_model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for img_in, img_target, _, _ in refit_val_loader:
            img_in, img_target = img_in.to(device), img_target.to(device)
            outputs = final_model(img_in)
            loss = final_criterion(outputs, img_target)
            val_loss += loss.item() * img_in.size(0)

    avg_val_loss = val_loss / len(refit_val_set)
    history_val_loss.append(avg_val_loss)

    if epoch % 10 == 0:
         print(f"Refit Epoch {epoch}/{REFIT_EPOCHS} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

refit_duration = time.time() - refit_start_time
print(f"Total Refit Time: {refit_duration:.2f} seconds")

plt.figure(figsize=(10, 5))
plt.plot(history_train_loss, label='Train Loss')
plt.plot(history_val_loss, label='Val Loss')
plt.title('Generator Refit (Standard + MSE): Loss Curve')
plt.legend()
plt.grid(True)
plt.show()

# ==========================================
# 5. JUDGE
# ==========================================
print(f"\n{'='*40}")
print("Training Judge")
print(f"{'='*40}")

j_train_size = int(0.9 * len(raw_train_dataset))
j_val_size = len(raw_train_dataset) - j_train_size
j_train_set, j_val_set = random_split(raw_train_dataset, [j_train_size, j_val_size])

j_train_loader = DataLoader(j_train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
j_val_loader = DataLoader(j_val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

class JudgePro(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1); self.bn1 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=0); self.bn2 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64 * 6 * 6, 128); self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool1(x)
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool2(x)
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return F.log_softmax(x, dim=1)

judge = JudgePro().to(device)
judge_optimizer = optim.Adam(judge.parameters(), lr=0.001)

for epoch in range(1, JUDGE_EPOCHS + 1):
    judge.train()
    correct_train, total_train = 0, 0
    for img, lbl in j_train_loader:
        img, lbl = img.to(device), lbl.to(device)
        judge_optimizer.zero_grad()
        out = judge(img)
        loss = F.nll_loss(out, lbl)
        loss.backward()
        judge_optimizer.step()
        _, pred = torch.max(out.data, 1)
        total_train += lbl.size(0)
        correct_train += (pred == lbl).sum().item()

    judge.eval()
    correct_val, total_val = 0, 0
    with torch.no_grad():
        for img, lbl in j_val_loader:
            img, lbl = img.to(device), lbl.to(device)
            out = judge(img)
            _, pred = torch.max(out.data, 1)
            total_val += lbl.size(0)
            correct_val += (pred == lbl).sum().item()

    print(f"Judge Epoch {epoch}/{JUDGE_EPOCHS} | Train Acc: {100*correct_train/total_train:.2f}% | Val Acc: {100*correct_val/total_val:.2f}%")

# ==========================================
# 6. VISUALIZATION (THRESHOLDING)
# ==========================================
print("\n--- VISUALIZATION ---")
test_loader_wrapped = DataLoader(test_dataset_wrapper, batch_size=BATCH_SIZE, shuffle=False)

judge.eval()
final_model.eval()

# Compute the final accuracy
correct, total = 0, 0
with torch.no_grad():
    for img_in, _, _, lbl_target in test_loader_wrapped:
        img_in, lbl_target = img_in.to(device), lbl_target.to(device)
        generated_imgs = final_model(img_in)
        judge_outputs = judge(generated_imgs)
        _, predicted = torch.max(judge_outputs.data, 1)
        total += lbl_target.size(0)
        correct += (predicted == lbl_target).sum().item()
print(f"Final Generator Accuracy: {100 * correct / total:.2f}%")

# Plotting with thresholding
indices = random.sample(range(len(raw_test_dataset)), 8)
fig, axs = plt.subplots(3, 8, figsize=(15, 6))

with torch.no_grad():
    for i, idx in enumerate(indices):
        img_in, _, lbl_in, lbl_target = test_dataset_wrapper[idx]
        img_in_gpu = img_in.unsqueeze(0).to(device)
        gen_img = final_model(img_in_gpu)

        # Thresholding
        sharp_img = (gen_img > 0.5).float()#IN THE EXPERIMENT WITHOUT IT I REMOVE THIS LINE AND USE gen_img INSTEAD OF sharp_img


        judge_out = judge(sharp_img)
        judge_pred = judge_out.argmax(dim=1).item()

        axs[0, i].imshow(img_in.squeeze(), cmap='gray')
        axs[0, i].set_title(f"In: {lbl_in}")
        axs[0, i].axis('off')

        axs[1, i].imshow(sharp_img.cpu().squeeze(), cmap='gray')
        col = 'green' if judge_pred == lbl_target else 'red'
        axs[1, i].set_title(f"Gen: {judge_pred}", color=col)
        axs[1, i].axis('off')

        dummy_bucket = test_dataset_wrapper.buckets[lbl_target]
        t_idx = random.choice(dummy_bucket)
        real_target_img, _ = raw_test_dataset[t_idx]
        axs[2, i].imshow(real_target_img.squeeze(), cmap='gray')
        axs[2, i].set_title(f"Targ: {lbl_target}")
        axs[2, i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
#FIXED TARGET (THRESHOLD)
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, SubsetRandomSampler, Dataset, random_split
from torchvision import datasets, transforms
from sklearn.model_selection import KFold
import numpy as np
import random
import time
import matplotlib.pyplot as plt

BATCH_SIZE = 256
NUM_WORKERS = 2
LEARNING_RATE = 1e-3
AE_EPOCHS = 20
REFIT_EPOCHS = 50
JUDGE_EPOCHS = 5
K_FOLDS = 5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ==========================================
# 1. DATASETS
# ==========================================
transform = transforms.ToTensor()
raw_train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
raw_test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

class NextDigitDataset(Dataset):
    def __init__(self, original_dataset):
        self.dataset = original_dataset
        self.buckets = {i: [] for i in range(10)}
        for idx in range(len(original_dataset)):
            _, label = original_dataset[idx]
            self.buckets[label].append(idx)

    def __getitem__(self, index):
        img_input, label_input = self.dataset[index]
        label_target = (label_input + 1) % 10

        # --- FIXED TARGET ---
        # For each input index we ALWAYS pick the same target.
        target_bucket = self.buckets[label_target]
        fixed_target_idx = target_bucket[index % len(target_bucket)]

        img_target, _ = self.dataset[fixed_target_idx]

        return img_input, img_target, label_input, label_target

    def __len__(self):
        return len(self.dataset)

full_train_dataset = NextDigitDataset(raw_train_dataset)
test_dataset_wrapper = NextDigitDataset(raw_test_dataset)

# ==========================================
# 2. MODEL: SUPER MLP
# ==========================================
class NextDigitMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 1024), nn.ReLU(),
            nn.Linear(1024, 512), nn.ReLU(),
            nn.Linear(512, 256), nn.ReLU(),
            nn.Linear(256, 128)
        )
        self.decoder = nn.Sequential(
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, 512), nn.ReLU(),
            nn.Linear(512, 1024), nn.ReLU(),
            nn.Linear(1024, 784), nn.Sigmoid()
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded.view(-1, 1, 28, 28)

def get_model():
    model = NextDigitMLP().to(device)
    optimizer = Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.BCELoss()
    return model, optimizer, criterion

# ==========================================
# 3. CROSS-VALIDATION
# ==========================================
kfold = KFold(n_splits=K_FOLDS, shuffle=True, random_state=42)

print(f"\n{'='*40}")
print(f"STARTING CV: FIXED TARGET EXPERIMENT")
print(f"{'='*40}")

fold_results = []

for fold, (train_ids, val_ids) in enumerate(kfold.split(raw_train_dataset)):
    print(f"\n>>> FOLD {fold + 1}/{K_FOLDS}")
    start_time = time.time()

    train_subsampler = SubsetRandomSampler(train_ids)
    val_subsampler = SubsetRandomSampler(val_ids)

    train_loader = DataLoader(full_train_dataset, batch_size=BATCH_SIZE, sampler=train_subsampler,
                              num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
    val_loader = DataLoader(full_train_dataset, batch_size=BATCH_SIZE, sampler=val_subsampler,
                            num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

    model, optimizer, criterion = get_model()
    scheduler = CosineAnnealingLR(optimizer, T_max=AE_EPOCHS)

    for epoch in range(1, AE_EPOCHS + 1):
        model.train()
        train_loss = 0.0
        for img_in, img_target, _, _ in train_loader:
            img_in, img_target = img_in.to(device), img_target.to(device)
            optimizer.zero_grad()
            outputs = model(img_in)
            loss = criterion(outputs, img_target)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * img_in.size(0)

        avg_train_loss = train_loss / len(train_ids)
        scheduler.step()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for img_in, img_target, _, _ in val_loader:
                img_in, img_target = img_in.to(device), img_target.to(device)
                outputs = model(img_in)
                loss = criterion(outputs, img_target)
                val_loss += loss.item() * img_in.size(0)

        avg_val_loss = val_loss / len(val_ids)

        if epoch % 5 == 0:
            print(f"    Epoch {epoch}/{AE_EPOCHS} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    fold_results.append(avg_val_loss)
    print(f"  >> Fold {fold+1} Finished. Time: {time.time()-start_time:.1f}s")

print(f"\nAVERAGE CV LOSS: {np.mean(fold_results):.4f}")

# ==========================================
# 4. REFIT
# ==========================================
print(f"\n{'='*40}")
print("REFIT: Training Final Model (Fixed Target)")
print(f"{'='*40}")

train_size = int(0.9 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size
refit_train_set, refit_val_set = random_split(full_train_dataset, [train_size, val_size])

refit_train_loader = DataLoader(refit_train_set, batch_size=BATCH_SIZE, shuffle=True,
                                num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
refit_val_loader = DataLoader(refit_val_set, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

final_model, final_optimizer, final_criterion = get_model()
final_scheduler = CosineAnnealingLR(final_optimizer, T_max=REFIT_EPOCHS)

history_train_loss = []
history_val_loss = []

refit_start_time = time.time()

for epoch in range(1, REFIT_EPOCHS + 1):
    final_model.train()
    train_loss = 0.0
    for img_in, img_target, _, _ in refit_train_loader:
        img_in, img_target = img_in.to(device), img_target.to(device)
        final_optimizer.zero_grad()
        outputs = final_model(img_in)
        loss = final_criterion(outputs, img_target)
        loss.backward()
        final_optimizer.step()
        train_loss += loss.item() * img_in.size(0)

    avg_train_loss = train_loss / len(refit_train_set)
    history_train_loss.append(avg_train_loss)
    final_scheduler.step()

    final_model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for img_in, img_target, _, _ in refit_val_loader:
            img_in, img_target = img_in.to(device), img_target.to(device)
            outputs = final_model(img_in)
            loss = final_criterion(outputs, img_target)
            val_loss += loss.item() * img_in.size(0)

    avg_val_loss = val_loss / len(refit_val_set)
    history_val_loss.append(avg_val_loss)

    if epoch % 10 == 0:
         print(f"Refit Epoch {epoch}/{REFIT_EPOCHS} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

refit_duration = time.time() - refit_start_time
print(f"Total Refit Time: {refit_duration:.2f} seconds")

plt.figure(figsize=(10, 5))
plt.plot(history_train_loss, label='Train Loss')
plt.plot(history_val_loss, label='Val Loss')
plt.title('Fixed Target Experiment: Loss Curve')
plt.legend()
plt.grid(True)
plt.show()

# ==========================================
# 5. JUDGE
# ==========================================
print(f"\n{'='*40}")
print("Training Judge")
print(f"{'='*40}")

j_train_size = int(0.9 * len(raw_train_dataset))
j_val_size = len(raw_train_dataset) - j_train_size
j_train_set, j_val_set = random_split(raw_train_dataset, [j_train_size, j_val_size])

j_train_loader = DataLoader(j_train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
j_val_loader = DataLoader(j_val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

class JudgePro(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1); self.bn1 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=0); self.bn2 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64 * 6 * 6, 128); self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool1(x)
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool2(x)
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return F.log_softmax(x, dim=1)

judge = JudgePro().to(device)
judge_optimizer = optim.Adam(judge.parameters(), lr=0.001)

for epoch in range(1, JUDGE_EPOCHS + 1):
    judge.train()
    correct_train, total_train = 0, 0
    for img, lbl in j_train_loader:
        img, lbl = img.to(device), lbl.to(device)
        judge_optimizer.zero_grad()
        out = judge(img)
        loss = F.nll_loss(out, lbl)
        loss.backward()
        judge_optimizer.step()
        _, pred = torch.max(out.data, 1)
        total_train += lbl.size(0)
        correct_train += (pred == lbl).sum().item()

    judge.eval()
    correct_val, total_val = 0, 0
    with torch.no_grad():
        for img, lbl in j_val_loader:
            img, lbl = img.to(device), lbl.to(device)
            out = judge(img)
            _, pred = torch.max(out.data, 1)
            total_val += lbl.size(0)
            correct_val += (pred == lbl).sum().item()

    print(f"Judge Epoch {epoch}/{JUDGE_EPOCHS} | Train Acc: {100*correct_train/total_train:.2f}% | Val Acc: {100*correct_val/total_val:.2f}%")

# ==========================================
# 6. VISUALIZATION (THRESHOLDING)
# ==========================================
print("\n--- VISUALIZATION (SHARP OUTPUT) ---")
test_loader_wrapped = DataLoader(test_dataset_wrapper, batch_size=BATCH_SIZE, shuffle=False)

judge.eval()
final_model.eval()

correct, total = 0, 0
with torch.no_grad():
    for img_in, _, _, lbl_target in test_loader_wrapped:
        img_in, lbl_target = img_in.to(device), lbl_target.to(device)
        generated_imgs = final_model(img_in)
        judge_outputs = judge(generated_imgs)
        _, predicted = torch.max(judge_outputs.data, 1)
        total += lbl_target.size(0)
        correct += (predicted == lbl_target).sum().item()
print(f"Final Generator Accuracy: {100 * correct / total:.2f}%")

indices = random.sample(range(len(raw_test_dataset)), 8)
fig, axs = plt.subplots(3, 8, figsize=(15, 6))

with torch.no_grad():
    for i, idx in enumerate(indices):
        img_in, _, lbl_in, lbl_target = test_dataset_wrapper[idx]
        img_in_gpu = img_in.unsqueeze(0).to(device)
        gen_img = final_model(img_in_gpu)

        # --- APPLY THRESHOLDING ---
        sharp_img = (gen_img > 0.5).float()

        judge_out = judge(sharp_img)
        judge_pred = judge_out.argmax(dim=1).item()

        axs[0, i].imshow(img_in.squeeze(), cmap='gray')
        axs[0, i].set_title(f"In: {lbl_in}")
        axs[0, i].axis('off')

        axs[1, i].imshow(sharp_img.cpu().squeeze(), cmap='gray')
        col = 'green' if judge_pred == lbl_target else 'red'
        axs[1, i].set_title(f"Gen: {judge_pred}", color=col)
        axs[1, i].axis('off')

        # Target (Reference)
        target_bucket = test_dataset_wrapper.buckets[lbl_target]
        fixed_t_idx = target_bucket[idx % len(target_bucket)]
        real_target_img, _ = raw_test_dataset[fixed_t_idx]

        axs[2, i].imshow(real_target_img.squeeze(), cmap='gray')
        axs[2, i].set_title(f"Ref: {lbl_target}")
        axs[2, i].axis('off')

plt.tight_layout()
plt.show()

EXPERIMENT: COMPARISON WITH PCA

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error
import numpy as np
import matplotlib.pyplot as plt
import time
import random

BATCH_SIZE = 256
N_COMPONENTS = 16  # I PUT A DIFFERENT VALUE HERE DEPENDING ON THE EXPERIMENT
JUDGE_EPOCHS = 5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ==========================================
# 1. DATASETS
# ==========================================
print("\n--- 1. LOADING DATA ---")
transform = transforms.ToTensor()
raw_train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
raw_test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# Flatten for PCA
X_train_numpy = raw_train_dataset.data.numpy().reshape(-1, 784) / 255.0
X_test_numpy = raw_test_dataset.data.numpy().reshape(-1, 784) / 255.0

# ==========================================
# 2. PCA TRAINING
# ==========================================
print(f"\n--- 2. RUNNING PCA (Components={N_COMPONENTS}) ---")
start_time = time.time()

pca = PCA(n_components=N_COMPONENTS)
pca.fit(X_train_numpy)

# Reconstruct the test set
X_test_pca_raw = pca.inverse_transform(pca.transform(X_test_numpy))

print(f"PCA Finished in {time.time() - start_time:.2f} seconds.")

# ==========================================
# 3. JUDGE TRAINING
# ==========================================
print(f"\n--- 3. TRAINING JUDGE (CNN) ---")

class JudgePro(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1); self.bn1 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=0); self.bn2 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64 * 6 * 6, 128); self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool1(x)
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool2(x)
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return F.log_softmax(x, dim=1)

judge = JudgePro().to(device)
judge_optimizer = optim.Adam(judge.parameters(), lr=0.001)
train_loader = DataLoader(raw_train_dataset, batch_size=BATCH_SIZE, shuffle=True)

for epoch in range(1, JUDGE_EPOCHS + 1):
    judge.train()
    correct, total = 0, 0
    for img, lbl in train_loader:
        img, lbl = img.to(device), lbl.to(device)
        judge_optimizer.zero_grad()
        out = judge(img)
        loss = F.nll_loss(out, lbl)
        loss.backward()
        judge_optimizer.step()
        _, pred = torch.max(out.data, 1)
        total += lbl.size(0)
        correct += (pred == lbl).sum().item()
    print(f"Judge Epoch {epoch}/{JUDGE_EPOCHS} | Acc: {100*correct/total:.2f}%")

# ==========================================
# 4. EVALUATION (RAW & THRESHOLD)
# ==========================================
print(f"\n--- 4. EVALUATING PCA ---")
judge.eval()

# Prepare the data
pca_tensor_raw = torch.tensor(X_test_pca_raw, dtype=torch.float32).view(-1, 1, 28, 28).to(device)
real_labels = raw_test_dataset.targets.to(device)

correct_raw, correct_thr = 0, 0
total_samples = 0

with torch.no_grad():
    for i in range(0, len(pca_tensor_raw), BATCH_SIZE):
        inputs_raw = pca_tensor_raw[i : i + BATCH_SIZE]
        labels = real_labels[i : i + BATCH_SIZE]

        # 1. Raw Evaluation
        out_raw = judge(inputs_raw)
        _, pred_raw = torch.max(out_raw.data, 1)
        correct_raw += (pred_raw == labels).sum().item()

        # 2. Threshold Evaluation (> 0.5)
        inputs_thr = (inputs_raw > 0.5).float()
        out_thr = judge(inputs_thr)
        _, pred_thr = torch.max(out_thr.data, 1)
        correct_thr += (pred_thr == labels).sum().item()

        total_samples += labels.size(0)

acc_raw = 100 * correct_raw / total_samples
acc_thr = 100 * correct_thr / total_samples

# Compute metrics (MSE & BCE)
total_mse = mean_squared_error(X_test_numpy, X_test_pca_raw)
# Clip for BCE because PCA gives values outside [0,1]
pca_clipped = np.clip(X_test_pca_raw, 1e-7, 1 - 1e-7)
t_orig = torch.tensor(X_test_numpy).float()
t_recon = torch.tensor(pca_clipped).float()
total_bce = F.binary_cross_entropy(t_recon, t_orig).item()

print(f"\n{'='*50}")
print(f"PCA RESULTS (Components: {N_COMPONENTS})")
print(f"{'='*50}")
print(f"MSE Loss                : {total_mse:.5f}")
print(f"BCE Loss                : {total_bce:.5f}")
print(f"{'-'*50}")
print(f"Judge Accuracy (Raw)    : {acc_raw:.2f}%")
print(f"Judge Accuracy (> 0.5)  : {acc_thr:.2f}%")
print(f"{'='*50}")

# ==========================================
# 5. VISUALIZATION
# ==========================================
indices = np.random.choice(len(X_test_numpy), 8, replace=False)
plt.figure(figsize=(15, 8))

with torch.no_grad():
    for i, idx in enumerate(indices):
        # Original
        ax = plt.subplot(3, 8, i + 1)
        plt.imshow(X_test_numpy[idx].reshape(28, 28), cmap='gray')
        if i == 3: ax.set_title("Input", fontsize=12)
        plt.axis('off')

        # PCA Raw
        ax = plt.subplot(3, 8, i + 1 + 8)
        img_raw = X_test_pca_raw[idx].reshape(28, 28)
        plt.imshow(img_raw, cmap='gray')
        if i == 3: ax.set_title("PCA Raw", fontsize=12)

        # Judge Prediction for Raw
        tens_raw = torch.tensor(img_raw).float().unsqueeze(0).unsqueeze(0).to(device)
        pred_r = judge(tens_raw).argmax().item()
        col_r = 'green' if pred_r == raw_test_dataset.targets[idx] else 'red'
        ax.text(0.5, -0.2, f"Pred: {pred_r}", color=col_r, ha='center', transform=ax.transAxes)
        plt.axis('off')

        # PCA Threshold
        ax = plt.subplot(3, 8, i + 1 + 16)
        img_thr = (img_raw > 0.5).astype(float)
        plt.imshow(img_thr, cmap='gray')
        if i == 3: ax.set_title("PCA Threshold", fontsize=12)

        # Judge Prediction for Threshold
        tens_thr = torch.tensor(img_thr).float().unsqueeze(0).unsqueeze(0).to(device)
        pred_t = judge(tens_thr).argmax().item()
        col_t = 'green' if pred_t == raw_test_dataset.targets[idx] else 'red'
        ax.text(0.5, -0.2, f"Pred: {pred_t}", color=col_t, ha='center', transform=ax.transAxes)
        plt.axis('off')

plt.tight_layout()
plt.show()